# SBMTeam final reproducible submission

**Team:** SBMTeam  
**Members:** Peretiatko Vsevolod (`sevaperetiatko`) and Platon Usachev
(`omnistudent`). Both members jointly developed the features, leakage-safe
validation, SVC kernels, ensembling and final model selection.

**Designated submission:** `submission_NEW5_sqrt_s100_aux20.csv`  
`final_submission.csv` reproduces that submission exactly. The same run also
recreates `submission_rescue_corrected_best.csv` as the team's second final
candidate.

This notebook performs only the frozen final training path. It contains no CV
search or experimental sweep, uses no internet or pretrained artifacts, and
reads competition files through relative paths.

## Reproducibility contract

- Fresh-kernel **Restart and Run All** is supported.
- Python, NumPy and model seeds are fixed to `0xFACED`.
- Only `train.csv` labels are used for fitting.
- `validation.csv` is scored after model selection and never changes weights.
- Test `row_id` is asserted against `sample_submission.csv` exactly.
- The total runtime is measured and asserted below 15 minutes.
- Final SVC predictions are produced immediately after fitting; no post-fit
  optimization is performed.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from final_model_runner import SEED, run

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('Seed:', SEED)
print('Working directory:', Path('.').resolve())

## Train the two frozen models and regenerate submissions

In [ ]:
result = run(output_dir='.')
print('Generated files:')
for path in result['paths']:
    print(' -', path.resolve())

## Required experiment table

In [ ]:
experiment_table = result['experiment_table'].copy()
display(experiment_table)
print('Timing breakdown:')
for name, value in result['timings'].items():
    print(f'{name}: {value:.3f}')

## Package and library versions

In [ ]:
version_table = pd.DataFrame(
    [{'package': name, 'version': version}
     for name, version in result['versions'].items()]
)
display(version_table)

## Final schema and identity audit

In [ ]:
final_submission = pd.read_csv('final_submission.csv')
designated_submission = pd.read_csv('submission_NEW5_sqrt_s100_aux20.csv')
rescue_submission = pd.read_csv('submission_rescue_corrected_best.csv')

assert list(final_submission.columns) == ['row_id', 'target']
assert np.array_equal(
    final_submission['row_id'].to_numpy(),
    designated_submission['row_id'].to_numpy(),
)
assert np.array_equal(
    final_submission['row_id'].to_numpy(),
    rescue_submission['row_id'].to_numpy(),
)
assert np.isfinite(final_submission['target']).all()
assert final_submission['target'].between(0.0, 1.0).all()

max_abs_designated_difference = float(np.max(np.abs(
    final_submission['target'].to_numpy()
    - designated_submission['target'].to_numpy()
)))
assert max_abs_designated_difference <= 1e-6

print('final_submission.csv rows:', len(final_submission))
print('max abs difference vs designated:', max_abs_designated_difference)
display(final_submission.head())